# 06. Free-VRAM-aware planner: формула, округление, ограничения

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

Разберём каждую строку get_enc/get_dec_optimal_tiling, затем проверим policy на сетке budgets без обращения к реальному CUDA GPU.

[Порядок и источники](../README.md) · [Связь с MIT Lab 3](../MIT_LAB3_BRIDGE.md) · [Все компоненты и методы](../SOURCE_MAP.md) · [Результаты](../reports/VALIDATION.md).

Работаем по циклу **формула → прогноз → код → измерение → вывод**. Все ячейки по умолчанию используют CPU; никакие веса, датасеты или ComfyUI jobs не скачиваются/запускаются. Исходные алгоритмы и учебные адаптеры явно различаются. Перед запуском численного опыта сформулируйте ожидаемый результат.


**MIT предпосылка:** Part 4–5 разделяет codec и latent model. Free-memory policy и GPU counters не являются задачами исходного Lab 3: здесь начинается deployment extension. [Карта границ](../MIT_LAB3_BRIDGE.md).


In [ ]:
from pathlib import Path
import sys, json, math
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
candidates=[Path.cwd(),*Path.cwd().parents]
COURSE=next((p/'vae' for p in candidates if (p/'vae/labkit').is_dir()),None)
assert COURSE is not None, 'Clone the repository first; see the Colab instructions in the course README. No weights are needed.'
sys.path.insert(0,str(COURSE))
from labkit.vae_lab import (NATIVE,TinyCodec,DiagonalPosterior,oracle_codec,plan,
    axis_coverage,anchored_starts,synthetic_video,errors,measure,show_source,
    source_hash,save_metrics,figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
LAB='06_free_vram_planner';metrics={}
print('Pinned source:',source_hash())


## 1. Что на самом деле моделирует heuristic

Память процесса состоит из weights, input/output, live activations, workspace, allocator reserve. При inference autograd не хранит training graph, но промежуточные tensors и full stitching buffers остаются:

$$
M_{peak}\approx M_{weights}+M_{IO}+M_{live}(Q_t,Q_h,Q_w)+M_{workspace}+M_{allocator}.
$$

В приложенном planner:

$$
A_{max}=\frac{M_{free}}{256\cdot17\cdot8},\qquad
N_{vals}=256\cdot17\cdot(H+32)(W+32).
$$

M_free в **bytes**. Чтобы размерность A была pixels, произведение denominator интерпретируется как эмпирические bytes на spatial pixel. 256,17,8 — константы реализации, а не универсально выведенный tensor liveness model. В формуле не используются B, dtype, latent channels, weights или optional attention. 8 нельзя без источника уверенно объявить bytes-per-element: BF16 сам по себе занимает 2 bytes.

Вторая проверка N_vals<2³¹ похожа на ограничение размера промежуточного tensor/indexing; код не документирует точную операцию. Не путать с лимитом RAM 2 GB.


In [ ]:
show_source('get_enc_optimal_tiling');show_source('get_dec_optimal_tiling')
shape=[1,3,121,480,864]
# Original CPU/MPS branch uses infinity, no memory API call.
for device in ['cpu','mps']:
    tile,stride=plan(shape,device=device)
    print(device,tile,stride)
    assert tile==(1,17,480,864)
metrics['non_cuda_policy']='infinite budget; temporal tile 17, no free-unified-memory query'


## 2. От числа tiles к двум осям

Если весь кадр не проходит, код выбирает:

$$
N=\max\left(\left\lceil HW/A_{max}\right\rceil,
\left\lceil N_{vals}/2^{31}\right\rceil\right),\qquad
k=\max(H/W,W/H).
$$

Пусть a·b≈N, b/a≈k. Подстановка b=ka даёт a≈sqrt(N/k), b≈sqrt(Nk). Обе величины округляются вверх. Большему размеру кадра назначается большее число splits. Это эвристика aspect ratio, не решение дискретной оптимизации peak memory или latency.

Для разделённой width:

$$
Q_w=8\left\lceil\frac{W}{8n_w}\right\rceil+16,\qquad S_w=Q_w-32.
$$

Получается nominal overlap 32 pixels. При слишком малом Q stride может стать нулевым/отрицательным. Округление tile может превысить short axis. Ни coverage, ни final memory bound после rounding код не доказывает. `get_dec` превращает latent shape обратно в sample dimensions (4(F_z−1)+1,8H_z,8W_z) и вызывает ту же encode heuristic; decoder cost profile отдельно не калибруется.


In [ ]:
rows=[]
for gib in [.125,.25,.5,1,2,4,8,16]:
    # Local torch namespace returns synthetic CUDA budget; no CUDA tensor/device is allocated.
    tile,stride=plan(shape,free_gib=gib,device='cuda:0')
    _,ft,ht,wt=tile;fs,hs,ws=stride
    hc,wc=axis_coverage(shape[3],ht,hs),axis_coverage(shape[4],wt,ws)
    rows.append({'synthetic_free_GiB':gib,'tile':list(tile),'stride':list(stride),
                 'height_missing':hc['missing'],'width_missing':wc['missing'],
                 'valid_spatial_grid':hc['valid'] and wc['valid'],
                 'tile_formula_GiB':256*17*8*ht*wt/2**30})
metrics['synthetic_budget_sweep']=rows
for row in rows:print(row)
fig,axes=plt.subplots(1,2,figsize=(10,3.2))
axes[0].plot([r['synthetic_free_GiB'] for r in rows],[r['tile'][2]*r['tile'][3] for r in rows],'o-');axes[0].set(xscale='log',xlabel='Synthetic CUDA free GiB',ylabel='Tile area',title='Original heuristic')
axes[1].plot([r['synthetic_free_GiB'] for r in rows],[max(r['height_missing'],r['width_missing']) for r in rows],'o-');axes[1].set(xscale='log',xlabel='Synthetic CUDA free GiB',ylabel='Missing edge pixels',title='Separate coverage postcondition')
fig.tight_layout();fig.savefig(figure_path(LAB,'planner_sweep'));plt.show()


## 3. Memory guard ≠ OOM guarantee

Сначала проверяются 0<S≤Q≤L, spatial/temporal alignment и coverage. Затем реальный прогон измеряет peak allocated/reserved; free_mem до запуска может измениться из-за других процессов. Практическая policy оставляет safety reserve, калибрует memory vs tile area, уменьшает tile при OOM и логирует retry. Нельзя зацикливать retries с тем же tile.

M3 Max 48 GB — **общая память**, включая macOS, CPU и GPU. `torch.mps.current_allocated_memory()` и `driver_allocated_memory()` описывают PyTorch/Metal allocations, а не «свободную VRAM 48 GB». `recommended_max_memory()` — рекомендованный working-set bound, не гарантированная free memory. Нужны отдельные measurements и консервативный user budget; CUDA policy не переносится автоматически.

В source все decoded tiles сохраняются в `rows` до concat; значит tiling снижает size одной активации, но не делает общую память постоянной при неограниченной duration/resolution. Список slices может удерживать backing storage. `empty_cache()` не освобождает live tensors.


In [ ]:
# Show a low-budget invalid-stride case without invoking any tensor generation.
low=plan([1,3,121,480,864],free_gib=.0001,device='cuda:0')
metrics['very_low_budget_policy']={'tile':list(low[0]),'stride':list(low[1])}
assert low[1][1]<=0 or low[1][2]<=0
print('Guard must reject:',low)
# Dense mask alone, if attention enabled: B * (Fz*Hz*Wz)^2 * bytes.
fz,hz,wz=31,60,108;n=fz*hz*wz
metrics['dense_attention_mask_fp32_GiB']=n*n*4/2**30
print('Possible full-latent dense mask GiB:',metrics['dense_attention_mask_fp32_GiB'])


## Упражнения и критерий перехода

1. Выведите sqrt factorization без подсказки и объясните ceiling.
2. Найдите input, где tile heuristic проходит memory approximation, но не coverage; запишите guard.
3. Спроектируйте calibration table (dtype,device,shape,tile,peak,time) и safety factor. Переход: не называйте synthetic budget реальным VRAM measurement.

Сначала запишите прогноз, затем меняйте один фактор. Разделяйте ошибку аппроксимации, ошибку интерфейса/формы и аппаратное ограничение. Сохранённые метрики относятся к этому запуску.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Источники

[HunyuanVideo, v6, §4.1 / Fig. 6 / Eq. 1, стр. 5–6](https://arxiv.org/pdf/2412.03603v6) · [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114) · [Официальный HunyuanVideo](https://github.com/Tencent-Hunyuan/HunyuanVideo) · [Закреплённый исследуемый исходник](../../kandinsky-6/reference/kandinsky-6/kandinsky/core/components/hunyuan_vae.py) · [Lecture 4 / исходный MIT Lab 3](../../lecture_04/README.md).

Математические выводы и малые опыты — авторские пояснения. Числа tiny-модели не являются оценкой pretrained HunyuanVideo/Kandinsky. Подробная provenance и ограничения — в SOURCE_MAP.
